# 편한길로 – YOLO 학습 (처음 하는 사람용)

**위에서부터 셀을 하나씩 실행**하면 됩니다 (셀 왼쪽 ▶ 버튼, 또는 `Shift + Enter`).

| 단계 | 하는 일 | 시간 |
|---|---|---|
| 0 | GPU 켜기 (아래 설명) | 1분 |
| 1~3 | 준비 (설치, 구글 드라이브 연결) | 3분 |
| 4 | Roboflow에서 계단·턱·경사로 사진 받기 | 5분 |
| 5 | 사진 합치기 | 1분 |
| 6 | **학습** (기다리기만 하면 됨) | 40분~1시간 |
| 7~8 | 점수 확인, 사진으로 확인 | 2분 |

### 0. GPU 켜기 (꼭 먼저)
메뉴 **런타임 → 런타임 유형 변경 → T4 GPU → 저장**

### Roboflow 키 넣기 (처음 한 번)
1. [roboflow.com](https://roboflow.com) 가입 → 오른쪽 위 프로필 → **Settings → API Keys** → Private API Key 복사
2. Colab 왼쪽 **🔑 (보안 비밀)** 아이콘 → **새 보안 비밀 추가**
3. 이름 `ROBOFLOW_API_KEY`, 값에 복사한 키 붙여넣기 → **노트북 액세스** 켜기

> 키를 코드 셀에 직접 붙여넣지 마세요. 노트북을 공유하면 키도 같이 퍼집니다.

## 1. GPU 확인
아래 결과에 `Tesla T4`가 보이면 성공. `command not found`가 나오면 0번(GPU 켜기)을 다시 하세요.

In [ ]:
!nvidia-smi --query-gpu=name,memory.total --format=csv

## 2. 프로그램 설치

In [ ]:
!pip -q install ultralytics roboflow

## 3. 구글 드라이브 연결
학습 결과를 드라이브에 저장합니다. Colab이 끊겨도 결과가 남습니다. 팝업이 뜨면 **허용**을 누르세요.

In [ ]:
from google.colab import drive
drive.mount("/content/drive")

BASE = "/content/drive/MyDrive/busan_project"   # 팀 공유 폴더 이름이 다르면 여기만 바꾸기
import os
os.makedirs(BASE, exist_ok=True)
print("저장 위치:", BASE)

## 4. 학습용 사진 받기 (Roboflow)

1. [Roboflow Universe](https://universe.roboflow.com)에서 `stairs`, `curb`, `wheelchair ramp`를 검색합니다.
2. 사진이 많고(수백 장 이상) **Object Detection**인 데이터셋을 고릅니다.
3. 데이터셋 페이지 주소에서 세 가지를 찾아 아래 `DATASETS`에 적습니다.

```
https://universe.roboflow.com/ine-4jp8h/staris/dataset/1
                             ^^^^^^^^^ ^^^^^^         ^
                             워크스페이스  프로젝트      버전
```

주소에 버전 숫자가 없으면 페이지의 **Dataset** 탭에서 버전을 고르면 주소에 나타납니다.
계단·턱·경사로 각각 1~2개씩, 모두 3~6개 정도면 충분합니다.

In [ ]:
# (워크스페이스, 프로젝트, 버전) — 직접 찾아서 채우기. 첫 줄은 예시라 버전이 다를 수 있음
DATASETS = [
    ("ine-4jp8h", "staris", 1),        # 계단
    # ("워크스페이스", "프로젝트", 1),   # 턱 (curb)
    # ("워크스페이스", "프로젝트", 1),   # 경사로 (ramp)
]

from google.colab import userdata
from roboflow import Roboflow

rf = Roboflow(api_key=userdata.get("ROBOFLOW_API_KEY"))
for ws, proj, ver in DATASETS:
    path = f"{BASE}/raw/{proj}"
    if os.path.exists(f"{path}/data.yaml"):
        print("이미 받음:", proj)
        continue
    try:
        rf.workspace(ws).project(proj).version(ver).download("yolov11", location=path)
        print("받음:", proj)
    except Exception as e:
        print(f"실패: {ws}/{proj}/{ver} → {e}")
        print("  주소의 워크스페이스·프로젝트·버전을 다시 확인하세요.")

## 5. 사진 합치기

데이터셋마다 클래스 이름이 달라서(`stair`, `staircase`, `caution_zone_stairs` …) 우리 클래스 3개로 통일합니다.
결과에 **'버린 클래스'**로 나온 이름 중 계단·턱·경사로가 있으면 `NAME_MAP`에 추가하고 다시 실행하세요.

In [ ]:
import random, shutil, yaml
from collections import Counter
from pathlib import Path

CLASSES = ["stairs", "curb", "ramp"]
NAME_MAP = {
    "stairs": "stairs", "stair": "stairs", "staircase": "stairs", "steps": "stairs",
    "caution_zone_stairs": "stairs",
    "curb": "curb", "kerb": "curb", "curbs": "curb",
    "ramp": "ramp", "ramps": "ramp", "wheelchair ramp": "ramp", "wheelchair_ramp": "ramp",
}

RAW = Path(BASE) / "raw"
OUT = Path("/content/dataset")          # 학습은 빠른 Colab 디스크에서
pairs, dropped = [], Counter()

for y in sorted(RAW.glob("*/data.yaml")):
    names = yaml.safe_load(open(y))["names"]
    if isinstance(names, dict):
        names = [names[k] for k in sorted(names, key=int)]
    names = [str(n).strip().lower() for n in names]
    id_map = {i: CLASSES.index(NAME_MAP[n]) for i, n in enumerate(names) if n in NAME_MAP}
    dropped.update(n for n in names if n not in NAME_MAP)
    for img in y.parent.glob("*/images/*"):
        lbl = img.parent.parent / "labels" / (img.stem + ".txt")
        if not lbl.exists():
            continue
        lines = []
        for line in open(lbl):
            p = line.split()
            if len(p) == 5 and int(p[0]) in id_map:      # 박스 라벨만 사용
                lines.append(" ".join([str(id_map[int(p[0])])] + p[1:]))
        if lines:
            pairs.append((img, lines))

random.seed(42)
random.shuffle(pairs)
n = len(pairs)
splits = {"train": pairs[:int(n * .7)], "valid": pairs[int(n * .7):int(n * .9)], "test": pairs[int(n * .9):]}

shutil.rmtree(OUT, ignore_errors=True)
boxes = Counter()
for s, items in splits.items():
    (OUT / s / "images").mkdir(parents=True)
    (OUT / s / "labels").mkdir(parents=True)
    for k, (img, lines) in enumerate(items):
        shutil.copy(img, OUT / s / "images" / f"{k:05d}{img.suffix}")
        (OUT / s / "labels" / f"{k:05d}.txt").write_text("\n".join(lines) + "\n")
        boxes.update(CLASSES[int(l.split()[0])] for l in lines)

yaml.safe_dump({"path": str(OUT), "train": "train/images", "val": "valid/images", "test": "test/images",
                "names": dict(enumerate(CLASSES))}, open(OUT / "data.yaml", "w"), sort_keys=False)

print("사진 수:", {s: len(v) for s, v in splits.items()})
print("클래스별 박스 수:", dict(boxes))
print("버린 클래스:", dict(dropped))
for c in CLASSES:
    if boxes[c] < 200:
        print(f"⚠️ {c} 박스가 {boxes[c]}개뿐 → 4번에서 {c} 데이터셋을 더 받으세요.")

## 6. 학습 ⏳

이제 기다리기만 하면 됩니다 (40분~1시간). 화면에 숫자가 계속 올라가면 정상입니다.

- `RUN_NAME`을 바꿀 때마다 새 실험으로 저장됩니다 (v1, v2 …).
- 중간에 Colab이 끊기면 → 1~3번 셀 다시 실행 → 아래 **이어서 학습** 셀 실행.

In [ ]:
from ultralytics import YOLO

RUN_NAME = "v1"            # 두 번째 실험은 "v2"
MODEL = "yolo11n.pt"       # v2에서는 "yolo11s.pt"로 바꿔 보기 (더 크고 느림)

model = YOLO(MODEL)
model.train(
    data="/content/dataset/data.yaml",
    epochs=50,             # 사진 전체를 50번 반복해서 보기
    imgsz=640,
    batch=16,
    patience=15,           # 15번 연속 좋아지지 않으면 일찍 끝냄
    project=f"{BASE}/runs",
    name=RUN_NAME,
    exist_ok=True,
)

In [ ]:
# 이어서 학습 (Colab이 끊겼을 때만 실행)
from ultralytics import YOLO
YOLO(f"{BASE}/runs/{RUN_NAME}/weights/last.pt").train(resume=True)

## 7. 점수 확인

**mAP50**이 핵심 점수입니다 (0~1, 높을수록 좋음).
- 0.5 이상: 쓸 만함
- 0.7 이상: 좋음

결과 줄을 `docs/실험_기록.md`에 그대로 옮겨 적으세요.

In [ ]:
best = YOLO(f"{BASE}/runs/{RUN_NAME}/weights/best.pt")
m = best.val(data="/content/dataset/data.yaml", split="test")
print(f"\n[{RUN_NAME}] mAP50 {m.box.map50:.3f} | 정밀도 {m.box.mp:.3f} | 재현율 {m.box.mr:.3f}")
for idx, ap in zip(m.box.ap_class_index, m.box.ap50):
    print(f"  {CLASSES[int(idx)]:8s} {ap:.3f}")

## 8. 사진으로 확인
시험용 사진 6장에 AI가 그린 박스를 보여줍니다.

In [ ]:
import glob
from IPython.display import Image, display

imgs = sorted(glob.glob("/content/dataset/test/images/*"))[:6]
for r in best.predict(imgs, conf=0.4, save=True, project="/content/pred", name=RUN_NAME, exist_ok=True):
    pass
for p in sorted(glob.glob(f"/content/pred/{RUN_NAME}/*"))[:6]:
    display(Image(p, width=480))

## 끝나면 팀에 공유할 것

드라이브 `busan_project/runs/<버전>/` 안에 다 저장돼 있습니다.

| 파일 | 쓰는 곳 |
|---|---|
| `weights/best.pt` | 학습된 AI. 부산 사진 탐지(`ai/04_detect.py`)에 사용 |
| `results.csv` | 포스터 그래프 1번 (학습 곡선) |
| `confusion_matrix.png`, `results.png` | 발표 자료 |